# Baseline: zero-shot direct answer, MMLU-ProX-Lite RU

Ноутбук скачивает модель, токенизатор и данные
бенчмарка напрямую с HuggingFace Hub (закреплённые revision). Единственное
требование — GPU с NF4-квантованием (рекомендуется A100 40GB; на CPU
инференс модели с 27 млрд параметров практически неприменим по времени).

## Задача

Оценка готовой (frozen) языковой модели на вопросах с выбором ответа на
русском языке — без дообучения, без демонстраций-примеров, без запроса
рассуждения перед ответом (zero-shot direct answer).


In [ ]:
%pip install -q "transformers>=4.46" accelerate bitsandbytes huggingface_hub pyarrow


## Конфигурация

Модель, ревизия, промпт и параметры генерации


In [ ]:
MODEL_ID = "Qwen/Qwen3.8-27B"
MODEL_REVISION = "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0"

DATA_ID = "li-lab/MMLU-ProX-Lite"
DATA_REVISION = "e82aafb9460529687d3c7e51b401d8dd1dd309dd"

INSTRUCTION = (
    "Выбери один правильный вариант ответа на вопрос. Ответь только "
    "в формате Ответ - (X), где X — латинская буква выбранного "
    "варианта. Не добавляй объяснение.\n"
)
QUESTION_PREFIX = "Вопрос:"
OPTIONS_PREFIX = "Варианты:"
ANSWER_REGEX = r"\AОтвет - (?:\(([A-J])\)|([A-J]))\Z"

MAX_NEW_TOKENS = 32
SEED = 20260913
SMOKE_SIZE = 10
# Полный прогон — все 588 вопросов, ~15 минут на A100. По умолчанию
# выключен, чтобы "Run All" не запускал его случайно. Поставьте True,
# когда smoke-тест ниже пройдёт нормально.
RUN_FULL = False


## Данные

Загружаем настоящий тестовый набор (588 вопросов, 14 предметных категорий) напрямую с HuggingFace, закреплённая revision

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id=DATA_ID, repo_type="dataset", revision=DATA_REVISION,
    filename="ru/test-00000-of-00001.parquet",
)
raw_rows = pq.read_table(path).to_pylist()


def normalize(row):
    options = [row[f"option_{i}"] for i in range(10)]
    answer = row["answer"]
    if answer not in "ABCDEFGHIJ" or len(answer) != 1:
        raise ValueError("Invalid answer")
    if row["answer_index"] != ord(answer) - ord("A"):
        raise ValueError("Answer index mismatch")
    return {
        "id": f"test:{row['question_id']}", "question": row["question"],
        "options": options, "answer": answer, "category": row["category"],
    }


rows = [normalize(r) for r in raw_rows]
assert len(rows) == 588
assert len({r["category"] for r in rows}) == 14
categories = {row["category"] for row in rows}
print(f"Загружено {len(rows)} вопросов, {len(categories)} категорий")


Загружено 588 вопросов, 14 категорий


## Промпт и извлечение ответа

Точно тот же формат: инструкция + вопрос + варианты, без демонстраций. Ответ модели должен строго соответствовать `Ответ - (X)` — иначе засчитывается как ошибка формата, попытки повтора не делаются.

In [ ]:
import re


def build_prompt(row):
    text = INSTRUCTION
    text += QUESTION_PREFIX + "\n" + row["question"] + "\n"
    text += OPTIONS_PREFIX + "\n"
    for i, option in enumerate(row["options"]):
        if option is not None:
            text += f"{chr(65 + i)}. {option}\n"
    return text


def extract(text, options, pattern):
    matches = [
        next(group for group in match.groups() if group is not None)
        for match in re.finditer(pattern, text.strip())
    ]
    selected = matches[0] if matches else None
    valid = selected is not None and options[ord(selected) - 65] is not None
    return {"extracted": selected, "valid_answer": valid,
            "distinct_matches": len(set(matches))}


print(build_prompt(rows[0]))


Выбери один правильный вариант ответа на вопрос. Ответь только в формате Ответ - (X), где X — латинская буква выбранного варианта. Не добавляй объяснение.
Вопрос:
Типичные регулирующие органы в сфере рекламы предлагают, например, чтобы реклама не должна: поощрять _________, вызывать ненужный ________ или _____, и не должна вызывать _______ оскорбление.
Варианты:
A. Безопасные практики, Страх, Зависть, Незначительное
B. Небезопасные практики, Дистресс, Радость, Незначительное
C. Безопасные практики, Желания, Зависть, Незначительное
D. Безопасные практики, Дистресс, Страх, Незначительное
E. Небезопасные практики, Желания, Зависть, Серьезное
F. Безопасные практики, Дистресс, Зависть, Серьезное
G. Безопасные практики, Желания, Страх, Серьезное
H. Небезопасные практики, Желания, Страх, Незначительное
I. Небезопасные практики, Дистресс, Страх, Серьезное


## Модель

Загрузка Qwen NF4 напрямую с HuggingFace Hub, закреплённая revision. Это самая тяжёлая ячейка (несколько минут на скачивание весов) и единственная, которой нужна GPU.

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Нужна GPU. В Colab: Runtime -> Change runtime type -> GPU (A100)."
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, device_map={"": 0},
    dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
)
model.eval().requires_grad_(False)
print(f"Модель загружена на {model.device}, словарь {len(tokenizer)} токенов")


## Генерация

Greedy, без сэмплирования, максимум 32 новых токена. Отдельно замеряем время до первого токена (TTFT) и среднее время между токенами (TPOT)

In [ ]:
import statistics
import time

from transformers import GenerationConfig


class TokenClock:
    """Отмечает момент прихода каждого сгенерированного токена."""

    def __init__(self):
        self.prompt_seen = False
        self.times = []

    def put(self, value):
        if not self.prompt_seen:
            self.prompt_seen = True
            return
        self.times.append(time.perf_counter())

    def end(self):
        pass


def generate_one(row):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": build_prompt(row)}],
        tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    count = inputs["input_ids"].shape[-1]
    eos = model.generation_config.eos_token_id or tokenizer.eos_token_id
    eos = [eos] if isinstance(eos, int) else eos
    generation_config = GenerationConfig(
        do_sample=False, num_beams=1, use_cache=True,
        max_new_tokens=MAX_NEW_TOKENS, eos_token_id=eos,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
    )
    clock = TokenClock()
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        result = model.generate(
            **inputs, generation_config=generation_config,
            tokenizer=tokenizer, streamer=clock,
        )
    torch.cuda.synchronize()
    end = time.perf_counter()
    ids = result[0, count:].tolist()
    text = tokenizer.decode(ids, skip_special_tokens=True).strip()
    ttft = clock.times[0] - start if clock.times else None
    tpot = (statistics.mean(
        [b - a for a, b in zip(clock.times, clock.times[1:])]
    ) if len(clock.times) > 1 else None)
    return {
        "id": row["id"], "category": row["category"], "target": row["answer"],
        "prediction": text, "input_tokens": count,
        "generated_tokens": len(ids), "e2e_latency_sec": end - start,
        "ttft_sec": ttft, "tpot_sec": tpot,
    }


## Smoke-тест

10 вопросов, чтобы убедиться, что формат ответа и пайплайн в целом работают, прежде чем запускать все 588.

In [ ]:
smoke_records = [generate_one(row) for row in rows[:SMOKE_SIZE]]
for record in smoke_records:
    check = extract(record["prediction"], rows[0]["options"], ANSWER_REGEX)
    print(f"{record['id']}: {record['prediction']!r} "
          f"-> valid_format={check['valid_answer']}, "
          f"e2e={record['e2e_latency_sec']:.2f}с")


## Полный прогон

588 вопросов один раз (`repeat=0`), без few-shot, без повторных попыток. Займёт около 15 минут на A100. Поставьте `RUN_FULL = True` выше и выполните ячейку.

In [ ]:
if not RUN_FULL:
    print("RUN_FULL=False — полный прогон пропущен. Поставьте True, чтобы "
          "выполнить все 588 вопросов.")
    records = smoke_records
else:
    records = []
    started = time.perf_counter()
    for i, row in enumerate(rows, 1):
        records.append(generate_one(row))
        if i % 50 == 0 or i == len(rows):
            print(f"{i}/{len(rows)}, прошло "
                  f"{time.perf_counter() - started:.0f}с", flush=True)


## Результат

Accuracy, доверительный интервал Уилсона (95%), доля распознанных ответов и распределение времени ответа.

In [ ]:
import math


def wilson_interval(correct, total):
    z = 1.959963984540054
    proportion = correct / total
    denominator = 1 + z * z / total
    center = (proportion + z * z / (2 * total)) / denominator
    half = (z * math.sqrt(proportion * (1 - proportion) / total
                          + z * z / (4 * total**2)) / denominator)
    return [center - half, center + half]


def percentile(values, probability):
    ordered = sorted(values)
    position = (len(ordered) - 1) * probability
    low, high = math.floor(position), math.ceil(position)
    return ordered[low] + (ordered[high] - ordered[low]) * (position - low)


by_id = {row["id"]: row for row in rows}
outcomes = []
for record in records:
    reference = by_id[record["id"]]
    answer = extract(record["prediction"], reference["options"], ANSWER_REGEX)
    outcomes.append({
        **answer,
        "correct": answer["valid_answer"]
        and answer["extracted"] == reference["answer"],
    })

correct = sum(o["correct"] for o in outcomes)
total = len(outcomes)
e2e = [r["e2e_latency_sec"] for r in records]

print(f"Accuracy: {correct}/{total} = {correct / total:.2%}")
ci = [round(x, 4) for x in wilson_interval(correct, total)]
extraction_rate = sum(o["valid_answer"] for o in outcomes) / total
print(f"Wilson 95% CI: {ci}")
print(f"Extraction rate: {extraction_rate:.2%}")
print(f"E2E mean / p95: {statistics.mean(e2e):.3f}с / "
      f"{percentile(e2e, 0.95):.3f}с")
ttft_values = [r["ttft_sec"] for r in records if r["ttft_sec"] is not None]
if ttft_values:
    print(f"TTFT mean: {statistics.mean(ttft_values):.3f}с")


## Сохранение результатов



In [ ]:
import json
import platform
from importlib import metadata
from pathlib import Path

environment = {
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "python": platform.python_version(),
    "packages": {
        name: metadata.version(name)
        for name in ("torch", "transformers", "accelerate",
                     "bitsandbytes")
    },
}

output = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "data_id": DATA_ID,
    "data_revision": DATA_REVISION,
    "run_full": RUN_FULL,
    "environment": environment,
    "count": total,
    "correct": correct,
    "accuracy": correct / total,
    "wilson_95_ci": ci,
    "extraction_rate": extraction_rate,
    "e2e_seconds": {
        "mean": statistics.mean(e2e),
        "p95": percentile(e2e, 0.95),
    },
    "ttft_seconds_mean": (
        statistics.mean(ttft_values) if ttft_values else None
    ),
    "records": records,
}

output_path = Path("baseline_results.json")
output_path.write_text(json.dumps(output, ensure_ascii=False, indent=2))
print(f"Сохранено: {output_path} ({output_path.stat().st_size / 1024:.0f} КБ)")

try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print("Не в Colab: файл остался в рабочей директории рантайма.")


## Ограничения

- Один seed, одна GPU-сессия — устойчивость числа по повторам не
  проверялась.
- Это не воспроизведение официальной таблицы бенчмарка (там — 5-shot
  с рассуждением перед ответом); числа между этими двумя протоколами
  не сравнимы напрямую.
- Baseline используется как опорная точка для сравнения с
  адаптированными вариантами входа.
